In [1]:
import sys
sys.path.append('../')

In [2]:
from sqlalchemy import inspect

In [3]:
import pandas as pd

In [4]:
from src.connections import get_postgres_engine, get_snowflake_engine, get_snowflake_engine2

In [5]:

snowflake_engine = get_snowflake_engine()
postgres_engine = get_postgres_engine()

In [6]:
with postgres_engine.connect() as conn:
    print("connected")

connected


In [7]:
#inspecting tables from postgres engine

tables = inspect(postgres_engine).get_table_names()
#tables

In [8]:
#creating dictionary for query

query = {}
for t in tables:
    query[t] = (f"select * from {t}")
    

In [112]:
#query

In [10]:
#standardizing table name for raw schema (prefix "stg_")

table_name = {}
for key,value in query.items():
    new = f"stg_{key}"
    table_name[new] = value
    


In [111]:
#table_name

In [22]:
#query

In [ ]:
#Read data from database 

In [12]:
dataframe = {}
for table, value in table_name.items():
    dataframe[table] = pd.read_sql(value,postgres_engine)

In [110]:
#dataframe['stg_reviews']

In [14]:
#writing to snowflake pd.to_sql(engine, table index =False)

for table, value in dataframe.items():
    value.to_sql(table,snowflake_engine,if_exists = 'replace', index = False)
print(f"loaded{table} sucessfully")

loadedstg_reviews sucessfully


In [ ]:
#Inspecting tables from snowflake before transformation

In [14]:
snow_tables = inspect(snowflake_engine).get_table_names()

In [16]:
#snow_tables

In [17]:
#standardizing table names to upper case 

query1 = {}
for t in snow_tables:
    query1[t] = (f"select * from {t.upper()}")
    

In [27]:
#query1

In [109]:
#query1

In [52]:
# extracting data from snowflake raw schema for tranformation

dataframe2 = {}
for table, value in query1.items():
    dataframe2[table] = pd.read_sql(value,snowflake_engine)

In [94]:
 #dataframe2['stg_customers']

In [22]:
#df1

In [ ]:
#from ydata_profiling import ProfileReport

In [108]:
#dataframe2['stg_customers'].info()

In [56]:
dataframe2['stg_customers'].shape

(128, 14)

In [48]:
#df1['last_name']

In [57]:
dataframe2['stg_customers']['full_name'] =dataframe2['stg_customers']['first_name']+" "+dataframe2['stg_customers']['last_name']

In [58]:
dataframe2['stg_customers'].drop(['first_name','last_name'],axis =1 , inplace =True)


In [59]:
dataframe2['stg_customers'].columns

Index(['id', 'email', 'phone', 'address', 'city', 'state', 'country',
       'postal_code', 'password', 'is_active', 'created_at', 'updated_at',
       'full_name'],
      dtype='object')

In [60]:
dataframe2['stg_customers'] = dataframe2['stg_customers'][['id','full_name','email','phone','address','city','state','country','postal_code','is_active','created_at','updated_at']]

In [30]:
#df1['city']=df1['city'].fillna('Not Given')

In [62]:
#dataframe2['stg_customers']

In [93]:
 #dataframe2['stg_products']


In [64]:
 dataframe2['stg_products'].columns

Index(['id', 'category_id', 'name', 'slug', 'description', 'price',
       'image_url', 'is_active', 'created_at', 'updated_at'],
      dtype='object')

In [65]:
 dataframe2['stg_products'].drop(['slug','image_url'],axis =1,inplace = True)

In [67]:

 #dataframe2['stg_products']

In [86]:
 #dataframe2['stg_order_items']


In [70]:
dataframe2['stg_categories']
dataframe2['stg_categories'].columns

Index(['id', 'name', 'slug', 'is_active', 'created_at', 'updated_at'], dtype='object')

In [87]:
dataframe2['stg_categories'].drop(['slug'],axis = 1, inplace =True)
#dataframe2['stg_categories']

KeyError: "['slug'] not found in axis"

In [88]:
#dataframe2['stg_coupons']


In [73]:
dataframe2['stg_coupons'].rename(columns={'discount_pct':'discount_percent'}, inplace = True)

In [75]:
#dataframe2['stg_coupons']

In [89]:
#dataframe2['stg_inventory']


In [90]:
#dataframe2['stg_orders']


In [91]:
#dataframe2['stg_payments']


In [80]:
#dataframe['stg_reviews']


In [107]:
#dataframe2['stg_reviews'].info()

In [92]:
#dataframe2['stg_shipping']


In [80]:
dataframe['stg_reviews']

In [96]:
#dataframe2

In [102]:
#prefixing silver layer table

transformed = {}
for table, value in dataframe2.items():
    new = f"trnsfrmed_{table.replace('stg_','')}"
    transformed[new] = value

In [104]:
#transformed

In [105]:
#snowflake engine activation
snowflake_eng2 = get_snowflake_engine2()

In [106]:
#loading transformed layer back to snowflake

for table, value in transformed.items():
     value.to_sql(table, snowflake_eng2, index = False, if_exists = 'replace')